## Lectura de datos 


In [0]:
csv_path = "/Volumes/workspace/ventas/bases/BigMart Sales.csv"
df = spark.read.csv(csv_path, header=True, inferSchema=True)
df.display() #mostrar la base de datos


## Subir JSON

In [0]:
json_path = "/Volumes/workspace/ventas/bases/drivers.json"
df_json = spark.read.format('json')\
    .option("inferSchema", True)\
    .option("multiline", False)\
    .load(json_path)
df_json.display()

In [0]:
df.printSchema() 

## Cargar librerias importantes

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
df_select = df.select(col("Item_Identifier"), col("Item_Weight"), col("Item_Fat_Content")).display()

In [0]:
df.select(col("Item_Identifier").alias("Item_ID")).display()

## Filter where

1. Filtrar datos con fat_content = Regular
2. Slice data type = soft_drink y weight menor a 10
3. Traer los datos con Tier en Tier 1 y Tier 2 y outlet size null.

In [0]:
df.filter(col("Item_Fat_Content") == "Regular").display()

In [0]:
df.filter((col("Item_Type") == "Soft Drinks") & (col("Item_Weight") < 10)).display()

In [0]:
df.filter((col("Outlet_Location_Type").isin("Tier 1", "Tier 2"))&(col("Outlet_Size").isNull())).display()

In [0]:
df.withColumnRenamed('Item_Weight', 'Item_Wt').display()

In [0]:
#Agregar columna
df.withColumn('flag',lit('new')).display()

In [0]:
# Multiplicar dos columnas y guardar el resultado en una columna nueva

df = df.withColumn('multiply', col('Item_Weight')*col('Item_MRP'))

df.display()

## Type Casting

In [0]:
df = df.withColumn('Item_Weight', col ('Item_Weight').cast(StringType()))
df.printSchema()

## Sort y OrderBy

In [0]:
df.sort(col("Item_Weight").asc()).display()

## Ordenar usando multiples columnas

In [0]:
df.sort(['Item_Weight', 'Item_Visibility'],asecending=[0,0]).display()

## Ascendente y otras descendente

In [0]:
df.sort(['Item_Weight','Item_Visibility'], ascending = [0,1])\
    .limit(10).display()

## Limit

Para ver cierta cantidad de resultados


In [0]:
df.limit(20).display()

## Drop

In [0]:
df.drop('Item_Visibility').display()

In [0]:
df.drop("Item_Visibility","Item_Type").display()

## Drop duplicates


In [0]:
df.dropDuplicates().display()

# Quitar duplicados de ciertas columnas

In [0]:
df.drop_duplicates(subset = ["Item_Type"]).display()

In [0]:
# Hace lo mismo que drop_duplicates
df.distinct().display()

## Union y UnionByName

## Preparamos la base de datos

In [0]:
data1 = [('1', 'kad'),
         ('2','sid')]
schema1 = 'id STRING', 'name String'

df1 = spark.createDataFrame(data=data1, schema=schema1)

data2 = [('3','Raul'),('4', 'Andrew')]
schema2 = 'id STRING', 'name String'

df2 = spark.createDataFrame(data=data2, schema=schema2)


In [0]:
df1.display()

In [0]:
df2.display()

In [0]:
df1.union(df2).display()

## Union by name

In [0]:
data1 = [('kad','1' ),
         ('sid','2')]
schema1 = 'name String', 'id STRING'

df1 = spark.createDataFrame(data=data1, schema=schema1)

In [0]:
df1.display()

In [0]:
df1.union(df2).display()

In [0]:
df1.unionByName(df2).display()

## DropNA

In [0]:
# quitar todos los NA's de la base, quitar todas observaciones que tengnan al menos un NA
df.dropna('any').display()

In [0]:
df.dropna('all').display()

## La mejor forma 

In [0]:
df.dropna(subset = ['Outlet_Size']).display()

## Fill NA

In [0]:
df.fillna('No disponible').display()

## Haciendolo por columna

In [0]:
df.fillna('No disponible', subset = ['Outlet_Size']).display()

## Split e Indexing

In [0]:
# Quiero separar Outlet_Type para usar split e indexing en esa columna
df.withColumn("Outlet_Type", split("Outlet_Type", " ")).display()

Indexing

In [0]:
df.withColumn("Outlet_Type", split("Outlet_Type", " ")[1]).display()

##GroupBy

In [0]:
df.groupBy('Item_Type').agg(sum('Item_MRP')).display()

In [0]:
df.groupBy('Item_Type').agg(avg('Item_MRP')).display() #promedio

## Con varias columnas

In [0]:
df.groupby('Item_Type','Outlet_Size').agg(sum("Item_MRP").alias("Total MRP")).display()

In [0]:
df.groupby('Item_Type','Outlet_Size').agg(sum("Item_MRP"),avg("Item_MRP"),count("Item_MRP")).display()

## Pivot

In [0]:
df.groupBy('Item_Type').pivot('Outlet_Size').agg(avg("Item_MRP")).display()

## When-otherwise

In [0]:
df = df.withColumn('vegetarianos', when(col('Item_Type') == 'Meat','No vegetariano').otherwise('vegetariano'))

display(df)

In [0]:
df.withColumn(
    'clasificar precio',
    when(
        (col('vegetarianos') == 'vegetariano') & (col("Item_MRP") < 100), 'barato'
    ).when(
        (col('vegetarianos') == 'No vegetariano') & (col("Item_MRP") > 100), 'caro'
    ).otherwise('medio')
).display()

## Joins

## Inner Join

## Preparamos los dataframes

In [0]:
data1 = [
    ('1', 'gaur', 'd01'),
    ('2', 'kit', 'd02'),
    ('3', 'sam', 'd03'),
    ('4', 'tim', 'd03'),
    ('5', 'aman', 'd05'),
    ('6', 'jim', 'd06')
]

schema1 = 'emp_id STRING, emp_name STRING, dept_id STRING'

df1 = spark.createDataFrame(data1, schema=schema1)

# ==========================
# SEGUNDO DATAFRAME
# ==========================
data2 = [
    ('d01', 'HR'),
    ('d02', 'Marketing'),
    ('d03', 'Accounts'),
    ('d04', 'IT'),
    ('d05', 'Finance')
]

schema2 = 'dept_id STRING, department STRING'

df2 = spark.createDataFrame(data2, schema=schema2)


In [0]:
df1.display()

In [0]:
df2.display()

## Inner Join

In [0]:
df1.join(df2,df1['dept_id']== df2['dept_id'],'inner').display()

## Left Join

In [0]:
df1.join(df2,df1['dept_id']== df2['dept_id'],'left').display()

In [0]:
df1.join(df2,df1['dept_id']== df2['dept_id'],'right').display()

## Antijoin

Nos da todas las observaciones que no tienen match en las 2 bases

In [0]:
df2.join(df1,df1['dept_id']== df2['dept_id'],'anti').display()